In [ ]:
import json
import pandas as pd

# 1. Load the required datasets
with open('data/stats_cons.json', 'r', encoding='utf-8') as f:
    stats_data = json.load(f)

with open('data/info_province.json', 'r', encoding='utf-8') as f:
    province_info = json.load(f)

# Create a mapping for Province IDs to Thai Names
prov_map = {p['prov_id']: p['province'] for p in province_info['province']}

summary_results = []

# 2. Iterate through each province in the statistics data
for prov in stats_data['result_province']:
    p_id = prov['prov_id']
    p_name = prov_map.get(p_id, p_id)
    
    # --- Constituency (MP App) Data ---
    to_mp = prov.get('turn_out', 0)
    inv_mp = prov.get('invalid_votes', 0)
    blk_mp = prov.get('blank_votes', 0)
    
    # Calculate percentages for MP App
    inv_mp_pct = (inv_mp / to_mp * 100) if to_mp > 0 else 0
    blk_mp_pct = (blk_mp / to_mp * 100) if to_mp > 0 else 0
    
    # --- Party List Data ---
    to_pl = prov.get('party_list_turn_out', 0)
    inv_pl = prov.get('party_list_invalid_votes', 0)
    blk_pl = prov.get('party_list_blank_votes', 0)
    
    # Calculate percentages for Party List
    inv_pl_pct = (inv_pl / to_pl * 100) if to_pl > 0 else 0
    blk_pl_pct = (blk_pl / to_pl * 100) if to_pl > 0 else 0
    
    # --- Calculate Absolute Difference in Turnout ---
    abs_diff = abs(to_mp - to_pl)
    
    # 3. Store the processed data
    summary_results.append({
        'จังหวัด': p_name,
        'มาใช้สิทธิ (เขต)': to_mp,
        'บัตรเสีย (เขต)': inv_mp,
        'บัตรเสีย (เขต %)': round(inv_mp_pct, 2),
        'ไม่ประสงค์ (เขต)': blk_mp,
        'ไม่ประสงค์ (เขต %)': round(blk_mp_pct, 2),
        'มาใช้สิทธิ (บัญชี)': to_pl,
        'บัตรเสีย (บัญชี)': inv_pl,
        'บัตรเสีย (บัญชี %)': round(inv_pl_pct, 2),
        'ไม่ประสงค์ (บัญชี)': blk_pl,
        'ไม่ประสงค์ (บัญชี %)': round(blk_pl_pct, 2),
        'Diff เขต vs บัญชี': abs_diff
    })

# 4. Create DataFrame and Sort by ABS Diff (Highest to Lowest)
df = pd.DataFrame(summary_results)
df_sorted = df.sort_values(by='Diff เขต vs บัญชี', ascending=False)

# 5. Output results
print("--- สรุปสถิติการเลือกตั้งแยกรายจังหวัด (เรียงตามค่าความเขย่ง ABS Diff) ---")
print(df_sorted.to_string(index=False))

# 6. Save to CSV for external use (Excel, etc.)
df_sorted.to_csv('output/thailand_election_stats_comparison.csv', index=False, encoding='utf-8-sig')
print("\n[Success] Data has been saved to 'thailand_election_stats_comparison.csv'")